# Deep dive analysis

In [0]:
%run /Repos/dung_nguyen_hoang@mfcgd.com/Utilities/Functions

In [0]:
import pyspark.sql.functions as F
import pandas as pd
from pyspark.sql.window import Window

lst_mthend='2025-06-30'
exrt_string = f'''
with xrt as (
select  cast(XCHNG_RATE as int) ex_rate,
        row_number() over (partition by XCHNG_RATE_TYP order by FR_EFF_DT DESC) rn
from    vn_published_cas_db.texchange_rates
where   XCHNG_RATE_TYP='U'
    and FR_CRCY_CODE='78'
    and to_date(FR_EFF_DT) <= '{lst_mthend}'
qualify rn=1
) select ex_rate from xrt
'''
exrt_df = sql_to_df(exrt_string, 1, spark)
ex_rate = exrt_df.collect()[0][0]
del exrt_df
print(ex_rate)

In [0]:
df_cols = [
    "POL_NUM", "CLI_NUM", "CHANNEL", "SUBMIT_TYPE", "BENEFIT_RIDER", "AIS_DECISION", "FINAL_UW_DECISION",
    "STP_FLAG", "BASE_APE_cat", "HIT_RULE", "ASSIGN_TEAM", "AIS_RULE_CATEGORY", "AIS_RULE_NAMES",
    "ADMIN_RULE_CATEGORY", "ADMIN_REMARKS", "DIGITEXX_REMARK_CATEGORY", "DIGITEXX_REMARKS",
    "TAT_SBMT_FUD", "TAT_FUD_ISSUE", "TAT_SBMT_ISSUE", "EXISTING_PO_IND", "EXISTING_LA_IND",
    "SBMT_DT", "ISSUE_DATE", "RIDER_APE_cat", "TOTAL_CLAIM_AMOUNT_cat", 'CLAIM_AMOUNT_PER_YEAR_cat', 
    'CLAIM_BENEFITS', 'CLAIM_MONTHS', 'MC_CLAIMS', 'MC_CLAIM_AMOUNT', 'HCR_CLAIMS', 'HCR_CLAIM_AMOUNT', 
    'FEMALE_CLAIMS', 'FEMALE_CLAIM_AMOUNT', 'OTHER_CLAIMS', 'OTHER_CLAIM_AMOUNT', 'VALID_HIS_CLM_IND',
    "AGENT_TIER", "AGE_LA_cat"
]

# Few columns for existing sales analysis
df_cols_2 = [
    "pol_num", "cli_num", "issue_date", "channel", "plan_code", "benefit_rider",
    "tot_ape", "base_ape", "rider_ape", "po_tenure",
    "ais_decision", "ais_rule_category", "ais_rule_names",
    "admin_rule_category", "admin_remarks", "digitexx_remarks", 
    "existing_po_ind", "existing_la_ind"
]

df_path = "/mnt/lab/vn/project/VN_NB/NB_UW/"

df = spark.read.format("parquet").load(df_path).filter(
    (F.col("POLICY_STATUS") == "APPROVED")
    #&(F.col("CHANNEL") == "AGENCY")
    &(F.col("SUBMIT_TYPE") != "SIO")
    &(F.col("image_date") <= lst_mthend)
).select(*[col.lower() for col in df_cols_2])

df = df.withColumn(
    "existing_ind",
    F.greatest(F.col("existing_po_ind"), F.col("existing_la_ind"))
)

df.createOrReplaceTempView("full_nb_pols")
# # Create a window specification to order by issue_date
# window_spec = Window.partitionBy("cli_num").orderBy(F.col("issue_date").desc())

# # Add a row number to each row within the cli_num partition
# df_with_row_num = df.withColumn("row_num", F.row_number().over(window_spec))

# # Cache the DataFrame with row numbers
# df_with_row_num.cache()

# # Filter to get the latest record for each cli_num
# df_dedup = df_with_row_num.filter(F.col("row_num") == 1).drop("row_num")
# df.limit(5).display()

In [0]:
# df.groupBy(
#     "channel"
# ).agg(
#     F.count("pol_num").alias("row_count"),
#     F.countDistinct("pol_num").alias("pol_count"),
#     F.countDistinct("cli_num").alias("cli_count")
# ).orderBy(
#     F.desc("channel")
# ).display()

# Analyis section

In [0]:
stp_df = spark.sql(f"""
SELECT  DISTINCT
        nb.*,
        CASE
            WHEN admin_remarks LIKE '%Client has claim history%' THEN "Y" ELSE "N"
        END AS claim_hx_ind,
        CONCAT_WS(', ', adr.ADDR_1, adr.ADDR_2, adr.ADDR_3, adr.ADDR_4) AS cli_address,
        COALESCE(
            prv.PROV_NM,
            regexp_replace(
                element_at(
                    split(regexp_replace(concat_ws(', ', adr.ADDR_1, adr.ADDR_2, adr.ADDR_3, adr.ADDR_4), '\\s*,\\s*', ','), ','),
                    size(split(regexp_replace(concat_ws(', ', adr.ADDR_1, adr.ADDR_2, adr.ADDR_3, adr.ADDR_4), '\\s*,\\s*', ','), ','))
                ),
                '\\.$',
                ''
            )
        ) AS province
FROM    full_nb_pols nb
    INNER JOIN
        vn_published_casm_cas_snapshot_db.tpolicys pol
        ON nb.pol_num = pol.pol_num
    INNER JOIN vn_published_casm_cas_snapshot_db.tclient_policy_links cpl 
        ON pol.POL_NUM = cpl.POL_NUM AND pol.image_date = cpl.image_date
    INNER JOIN vn_published_cas_db.tclient_addresses adr 
        ON cpl.CLI_NUM = adr.CLI_NUM AND cpl.ADDR_TYP = adr.ADDR_TYP
    LEFT JOIN vn_published_cas_db.tprovinces prv 
        ON substr(adr.ZIP_CODE, 1, 2) = prv.PROV_ID
    LEFT JOIN
        vn_aa_reports.pre_approved_exclusion exc
        ON nb.cli_num = exc.cli_num
WHERE   1=1
    --  Policy must still be in-force as of snapshot date
    AND pol.image_date = '{lst_mthend}'
    --  Exclude Decline/Postpone/Rescinded/Rating/Exclusion & Medical declaration
    AND (exc.excl_ind = 'N' AND exc.rating_his_ind = 'N' AND exc.decl_ind = 'N' AND exc.med_his_ind = 'N')
    --  Exclude Medical check-up
    AND LOWER(nb.ais_rule_category) NOT LIKE '%medical%'
    AND LOWER(nb.admin_rule_category) NOT LIKE '%medical%'
    AND cpl.LINK_TYP = 'O'
    AND cpl.REC_STATUS = 'A'
""")

#print(stp_df.select("cli_num").distinct().count())

### Load claims with ICD codes

In [0]:
file_path = "/dbfs/mnt/lab/vn/project/cpm/LEGO/Pre_approved_2/acute_icd_list.csv"
df = pd.read_csv(file_path)

# Convert the first column into a list of strings
acute_list = df.iloc[:, 0].astype(str).tolist()

# Convert to string
acute_str = ",".join(f"'{code}'" for code in acute_list)

# Print the result
# print(f"acute_str = ({acute_str})")

In [0]:
clm_raw = spark.sql(f"""
-- Exact all Approved claim amounts past 2 years
with clm_type as (
select  cast(fld_valu as int) as clm_code
        , fld_valu_desc as benefit
        ,case when cast(fld_valu as int) in (3,7,36) then 'Quyền lợi trợ cấp-Medicash'
          when cast(fld_valu as int) in (11,27,28,29,31,40,41,42,50,51) then 'Quyền lợi điều trị-Healthcare'
          when cast(fld_valu as int) in (9,38,45) then 'Quyền lợi thai sản-Female Benefits' 
          when cast(fld_valu as int) in (5,21,23,24,25,30,33,34,35) then 'Quyền lợi bệnh lý nghiêm trọng-Major Diseases'
          when cast(fld_valu as int) in (4,20,37) then 'Quyền lợi tử vong-Major Death'
          else 'Quyền lợi lớn khác-Other Major Benefits'
     end as clm_benefit_type
from    vn_published_cas_db.tfield_values
where   fld_nm='CLM_CODE'
), chronic_list as (
select
        Medical_History_Id__c
        ,concat_ws('/',collect_list(Disease_Code__c)) icd_codes
        ,concat_ws('/',collect_list(Disease_Name__c)) icd_names
        ,'Chronic' as icd_type
from
        vn_published_sfdc_easyclaims_db.diagnosis__c
where  Disease_Code__c is not null
   and Disease_Code__c not in ({acute_str})
group by
        Medical_History_Id__c
), acute_list as (
select
        Medical_History_Id__c
        ,concat_ws('/',collect_list(Disease_Code__c)) icd_codes
        ,concat_ws('/',collect_list(Disease_Name__c)) icd_names
        ,'Acute' as icd_type
from
        vn_published_sfdc_easyclaims_db.diagnosis__c
where  Disease_Code__c is not null
   and Disease_Code__c in ({acute_str})
group by
        Medical_History_Id__c
),
-- Extract all claims approved before Jul'25
all_claims as (
select  clm_ref.POL_NUM
        ,clm_ref.CLI_NUM
        ,clm.Case_Number__c
        ,clm.Name as claimID
        ,med.Id
        ,cast(clm_ref.CLM_CODE as int) as CLM_CODE
        ,case
            when clm_ref.direct_billing = 'Y'
                then 'Direct Billing'
            when clm_ref.src_cd = 'CWS'
                then 'CWS'      
            when sfo.origin = 'Easy Claims'
                then 'Easy Claims'
            when sfo.origin = 'Branch Submission'
                then 'Branch Submission'
            when clm_ref.req_id is null then
                case
                    when sfo.origin is null then 'Branch Submission'
                    else
                        case sfo.origin
                            when 'Easy Claims' then 'Easy Claims'
                            when 'Branch Submission' then 'Branch Submission'
                            else
                            'Branch Submission'
                        end
                end
            when sfo.origin is null and clm_ref.req_id is not null
                then 'Easy Claims'
            else 'Branch Submission'
        end submit_channel
        ,to_date(clm_ref.CLM_EFF_DT) as clm_dt
        ,substr(to_date(clm_ref.CLM_EFF_DT),1,7) as clm_mth
        ,to_date(clm_ref.CLM_APROV_DT) as clm_aprv_dt
        ,substr(to_date(clm_ref.CLM_APROV_DT),1,7) as clm_aprv_mth
        ,case when clm_ref.CLM_STAT_CODE='A' then 'Accepted'
             when clm_ref.CLM_STAT_CODE='D' then 'Denied'
             else 'In-progress'
        end as clm_status
        ,clm_ref.CLM_APROV_AMT

from    vn_published_sfdc_easyclaims_db.claim__c clm left join
        vn_published_sfdc_easyclaims_db.`case` sfo on (clm.Case_Number__c = sfo.CaseNumber) left join
        vn_published_sfdc_easyclaims_db.medical_history__c med on (clm.Id = med.Claim__c) left join
        vn_published_cas_db.tclaim_details clm_ref on (clm.Case_Number__c = clm_ref.req_id)
where   1=1
    and sfo.Status <> 'Closed at Submission'
    and to_date(clm_ref.CLM_APROV_DT) <= '{lst_mthend}'
)
select  clm.CLI_NUM
        , clm.POL_NUM
        --, clm.Case_Number__c
        , clm.claimID
        --, clm.Id
        , typ.clm_benefit_type
        , clm.submit_channel
        , clm.clm_dt
        , clm.clm_mth
        , clm.clm_aprv_dt
        , clm.clm_aprv_mth
        , clm.clm_status        
        -- Total Claim amount from HC & MC only
        ,cast(
            case when clm.CLM_CODE in (3,7,36,11,27,28,29,31,40,41,42,50,51,9,38,45) then CLM_APROV_AMT end
            as int
        ) as minor_clm_amt
        -- Total Claim amount in the last 2 years
        ,cast(
            case when clm_dt > '2023-06-30' then CLM_APROV_AMT end
            as int
        ) as clm_amt_last_2yrs
        , coalesce(chr.icd_type,acu.icd_type,'None') as icd_type 
        , concat_ws('/',chr.icd_codes,acu.icd_codes) as icd_codes
        , concat_ws('/',chr.icd_names,acu.icd_names) as icd_names
from    all_claims clm 
    left join
        chronic_list chr on (clm.Id = chr.Medical_History_Id__c) 
    left join
        acute_list acu on (clm.Id = acu.Medical_History_Id__c)
    left join
        clm_type typ on (clm.CLM_CODE = typ.CLM_CODE)
where   1=1 
""").distinct()

# check_dup(clm_raw, "claimID")

clm_dtl = clm_raw.groupBy(
    'cli_num',
).agg(
    #F.concat_ws('/',F.collect_set(F.col('claimant_role'))).alias('claimant_role'),
    F.countDistinct('claimID').alias('no_claims'),
    #F.concat_ws('/',F.collect_set(F.col('claimID'))).alias('claim_list'),
    F.min(F.col('clm_dt')).alias('first_clm_dt'),
    F.max(F.col('clm_dt')).alias('last_clm_dt'),
    F.min(F.col('clm_aprv_dt')).alias('first_clm_aprv_dt'),
    F.max(F.col('clm_aprv_dt')).alias('last_clm_aprv_dt'),
    F.sum(F.col('minor_clm_amt')).alias('total_minor_clm_amt'),
    F.sum(F.col('clm_amt_last_2yrs')).alias('total_clm_amt_last_2yrs'),
    F.concat_ws('/',F.collect_set(F.col('clm_benefit_type'))).alias('clm_benefit_type'),
    #F.concat_ws('/',F.collect_set(F.col('submit_channel'))).alias('submit_channel'),
    F.concat_ws('/',F.collect_set(F.col('icd_type'))).alias('icd_type'),
    F.concat_ws('/',F.collect_set(F.col('icd_codes'))).alias('icd_codes'),
    F.concat_ws('/',F.collect_set(F.col('icd_names'))).alias('icd_names'),
    #F.max(F.col('exclude_ind')).alias('exclude_ind')
).orderBy('cli_num')

clm_dtl = clm_dtl.withColumn(
    "hc_mc_clm_only_ind",
    F.when(
        ~F.col("clm_benefit_type").like("%Female%") &
        ~F.col("clm_benefit_type").like("%Major%")
    , "Y")
    .otherwise("N")
)

# clm_dtl.limit(5).display()
# check_dup(clm_dtl, "cli_num")

In [0]:
cols_order = [
    "pol_num", "cli_num", "issue_date", "plan_code",
    "cli_address", "province", "pol_tenure", "benefit_rider",
    "claim_hx_ind", "no_claims", "clm_benefit_type",
    "last_clm_dt", "icd_type", "icd_codes", 
    "hc_mc_clm_only_ind", "total_minor_clm_amt", "total_clm_amt_last_2yrs",
    "icd_names", "tot_ape", "base_ape", "rider_ape",
    "admin_rule_category"
]

result_df = stp_df.join(
    clm_dtl,
    on="cli_num",
    how="left"
).fillna(
    {
        "no_claims": 0, "total_minor_clm_amt": 0, "total_clm_amt_last_2yrs": 0
    }
)

result_df = result_df.withColumn(
    "po_tenure",
    F.floor(F.months_between(F.lit(lst_mthend), F.col("issue_date")))
).withColumnRenamed(
    "po_tenure", "pol_tenure"
).select(
    *cols_order
).withColumn(
    "image_date",
    F.to_date(F.lit(lst_mthend))
)

# check_dup(result_df, "pol_num")
# result_df.filter(
#     F.col("no_claims") > 0
# ).limit(5).display()

In [0]:
# result_df.write.mode(
#     "overwrite"
# ).option(
#     "partitionOverwriteMode", "dynamic"
# ).partitionBy(
#     "image_date"
# ).parquet(
#     "/mnt/lab/vn/project/VN_NB/NB_CLAIMS/"
# )

# Save the DataFrame as a Parquet file (or another format if needed)
result_df.write.mode(
    "overwrite"
).option(
    "overwriteSchema", "true"
).partitionBy(
    "image_date"
).saveAsTable(
    "vn_aa_reports.vn_nb_claims_analysis"
)

### Save results

In [0]:
result = result_df.toPandas()
result.to_csv(
    "/dbfs/mnt/lab/vn/project/VN_NB/CSV_FILES/vn_nb_claims_history.csv",
    index=False,
    header=True,
    encoding='utf-8-sig'
)

print(result.shape)